In [1]:
!pip install mediapipe

In [ ]:
# ---------------- STEP 1: Import Libraries ----------------
import cv2                    
import numpy as np
import pyautogui             
import time                    
import os
import urllib.request
import mediapipe as mp        

# ---------------- STEP 2: MediaPipe Tasks Setup ----------------
# Menggunakan Tasks API modern untuk bypass bug inisialisasi versi Python 3.13
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

model_path = 'hand_landmarker.task'
if not os.path.exists(model_path):
    print("Mengunduh model pelacak tangan dari Google...")
    url = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
    urllib.request.urlretrieve(url, model_path)

# Mengatur opsi detektor (Maksimal 1 tangan dengan akurasi deteksi 70%)
base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.HandLandmarkerOptions(base_options=base_options, num_hands=1)
detector = vision.HandLandmarker.create_from_options(options)

# ---------------- STEP 3: Open Webcam ----------------
cap = cv2.VideoCapture(0)                     # Mengaktifkan kamera utama laptop

# ---------------- STEP 4: Cooldown Setup ----------------
prev_action_time = 0                          # Mencatat waktu perintah terakhir
cooldown = 1.0                                # Batas jeda perintah (1 detik)
status_text = "Waiting for gesture..."        # UI teks awal status sistem

# ---------------- STEP 5: Finger Counting Function ----------------
def count_fingers_tasks(hand_landmarks_list):
    if not hand_landmarks_list:
        return 0
        
    landmarks = hand_landmarks_list[0]
    # ID Titik ujung jari: Telunjuk (8), Tengah (12), Manis (16), Kelingking (20) [cite: 35, 101]
    tips = [8, 12, 16, 20]     
    fingers = 0               

    # 1. Deteksi 4 Jari Atas (Sumbu Y)
    for tip in tips:
        if landmarks[tip].y < landmarks[tip - 2].y:
            fingers += 1      

    # 2. Deteksi Ibu Jari (Sumbu X) [cite: 35, 101]
    # Jika ujung jempol (4) berada di sebelah kiri joint (3), berarti jempol terbuka/terangkat
    if landmarks[4].x < landmarks[3].x:
        fingers += 1          

    return fingers 

# ---------------- STEP 6: Main Loop ----------------
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Membalikkan gambar horizontal agar gerakan tangan seperti cermin (mirroring)
    frame = cv2.flip(frame, 1)
    h_frame, w_frame, _ = frame.shape

    # Membuat kotak pembatas eksekusi di area tengah-atas layar agar fokus ke tangan
    roi_top, roi_bottom, roi_left, roi_right = 60, 380, 200, 520
    cv2.rectangle(frame, (roi_left, roi_top), (roi_right, roi_bottom), (0, 255, 0), 2)
    
    # Memotong citra frame hanya di area dalam kotak ROI
    roi = frame[roi_top:roi_bottom, roi_left:roi_right]
    h_roi, w_roi, _ = roi.shape

    # 1. Konversi area kotak ROI dari ruang warna BGR ke HSV
    hsv_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    
    # 2. Atur ambang batas warna kulit manusia (Skin Color Range)
    lower_skin = np.array([0, 20, 70], dtype=np.uint8)
    upper_skin = np.array([20, 255, 255], dtype=np.uint8)
    
    # 3. Thresholding: Membuat masker biner (Kulit = PUTIH, Objek lain = HITAM)
    skin_mask = cv2.inRange(hsv_roi, lower_skin, upper_skin)
    
    # 4. Reduksi Noise dengan Gaussian Blur sebelum penarikan tepi kontur [cite: 677, 721]
    blurred_mask = cv2.GaussianBlur(skin_mask, (5, 5), 1)
    
    # 5. Canny Edge Detection untuk visualisasi kontur luar struktur kulit tangan [cite: 667, 709]
    canny_edges = cv2.Canny(blurred_mask, 30, 90)
    
    # 6. Bitwise AND: Bersihkan background ROI asli, sisakan hanya objek berwarna kulit
    isolated_skin_rgb = cv2.bitwise_and(roi, roi, mask=blurred_mask)
    # Konversi ke RGB 
    isolated_skin_rgb = cv2.cvtColor(isolated_skin_rgb, cv2.COLOR_BGR2RGB)

    # Kirim citra ROI yang sudah terisolasi bersih dari gangguan benda mati ke MediaPipe AI
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=isolated_skin_rgb)
    result = detector.detect(mp_image)
    current_time = time.time()

    # ---------------- STEP 7 & 8: Jika Tangan Terdeteksi di Dalam ROI ----------------
    if result.hand_landmarks:
        # Menggambar koordinat titik landmark internal tangan di layar utama (Transformasi koordinat lokal ke global) [cite: 35, 101]
        for lm in result.hand_landmarks[0]:
            cx = int(lm.x * w_roi) + roi_left
            cy = int(lm.y * h_roi) + roi_top
            cv2.circle(frame, (cx, cy), 5, (0, 255, 0), -1)

        # Hitung jumlah jari aktif terangkat (1 - 5)
        finger_count = count_fingers_tasks(result.hand_landmarks)

        # ---------------- LOGIKA EKSEKUSI PERINTAH POWERPOINT ----------------
        if (current_time - prev_action_time) > cooldown:

            # 1 JARI -> Slide Selanjutnya
            if finger_count == 1:
                pyautogui.press("right")
                status_text = "Next Slide"
                prev_action_time = current_time

            # 2 JARI -> Slide Sebelumnya
            elif finger_count == 2:
                pyautogui.press("left")
                status_text = "Previous Slide"
                prev_action_time = current_time

            # 3 JARI -> Mulai Tampilan Presentasi Layar Penuh (F5)
            elif finger_count == 3:
                pyautogui.press("f5")
                status_text = "Start Slideshow"
                prev_action_time = current_time

            # 4 JARI -> Keluar dari Mode Layar Penuh (ESC)
            elif finger_count == 4:
                pyautogui.press("esc")
                status_text = "Exit Slideshow"
                prev_action_time = current_time

            # 5 JARI -> Menutup Aplikasi PowerPoint (ALT + F4)
            elif finger_count == 5:
                pyautogui.hotkey("alt", "f4")
                status_text = "Close PowerPoint"
                prev_action_time = current_time
    else:
        status_text = "Waiting for hand..."

    # ---------------- STEP 9 & 10: Teks UI & Windows Output ----------------
    cv2.putText(frame, status_text, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)
    if result.hand_landmarks:
        cv2.putText(frame, f"Fingers: {finger_count}", (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    # Jendela Jendela Tampilan Hasil Pemrosesan
    cv2.imshow("Hand Controlled PowerPoint (Main Window)", frame)
    cv2.imshow("Isolasi Kulit Sesi 01 (HSV Skin Mask)", skin_mask)
    cv2.imshow("Deteksi Tepi Sesi 03 (Canny Edge)", canny_edges)

    # STEP 11: Tekan Tombol q untuk Keluar dari Program
    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break

# ---------------- STEP 12: Release Resources ----------------
cap.release()
cv2.destroyAllWindows()